# RetainIQ 01 - Data cleaning
*See*: what is wrong with the raw file and what we did about it. Every action is logged; nothing is deleted silently.

In [1]:
import sys, json, pandas as pd
sys.path.insert(0, '../src')
from IPython.display import Image, display, Markdown
pd.set_option('display.width', 200)

In [2]:
import clean
clean.main()

clean_frame: 10000 -> 10000 rows
[PASS] row count == 10,000
[PASS] no duplicate IDs
[PASS] no text in numeric columns
[PASS] no missing values
[PASS] balance max < 2M (no summary rows)
[PASS] tenure max <= 82
[PASS] churn in {0,1}
                                              issue  rows_affected                                                              action
0                            Junk rows above header              1                                                             dropped
1                   Columns outside data dictionary              2                                        dropped ['aging', '<empty>']
2   Non-customer rows (pasted summary/pivot tables)             18                                                             dropped
3   Numbers stored as text with thousand separators           9895                                 separators stripped, cast to number
4                              Exact duplicate rows              0                            

,customer_id,age,gender,region,customer_segment,tenure_years,monthly_income,avg_monthly_balance,num_products,has_credit_card,...,has_complaint,log_days_inactive,inactive_45plus,inactivity_band,low_engagement,rm_eligible_unassigned,segment_rank,age_band,complaint_band,product_band
0,CUST100000,18.0,Female,North,Mass Market,10.7,3051.0,4485.0,2.0,0.0,...,0.0,2.833213,0.0,16-45,0.0,0.0,0,18-25,0,2
1,CUST100001,42.0,Female,South,Mass Affluent,0.4,6329.0,12972.0,2.0,1.0,...,1.0,1.098612,0.0,<=15,0.0,0.0,1,36-45,1,2
2,CUST100002,34.0,Female,Metro,Mass Market,5.2,4292.0,4356.0,2.0,1.0,...,0.0,4.584967,1.0,90+,0.0,0.0,0,26-35,0,2
3,CUST100003,27.0,Female,East,Mass Market,1.3,3335.0,4026.0,1.0,0.0,...,0.0,4.158883,1.0,46-90,0.0,0.0,0,26-35,0,1
4,CUST100004,18.0,Male,West,Mass Affluent,1.9,7453.0,13083.0,2.0,0.0,...,0.0,4.753590,1.0,90+,0.0,0.0,1,18-25,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,CUST109995,50.0,Male,North,Mass Market,5.4,3423.0,2243.0,1.0,0.0,...,0.0,3.951244,1.0,46-90,0.0,0.0,0,46-55,0,1
9996,CUST109996,60.0,Female,South,Mass Affluent,15.4,4277.0,29188.0,3.0,0.0,...,1.0,2.995732,0.0,16-45,0.0,1.0,1,56-65,1,3
9997,CUST109997,25.0,Female,Metro,Mass Market,2.9,2271.0,4498.0,2.0,0.0,...,0.0,3.091042,0.0,16-45,1.0,0.0,0,18-25,0,2
9998,CUST109998,47.0,Female,South,Affluent,1.2,11887.0,65987.0,4.0,0.0,...,0.0,2.772589,0.0,<=15,0.0,0.0,2,46-55,0,4


In [3]:
pd.read_csv('../outputs/data_cleaning_log.csv')

,issue,detection_rule,rows_affected,action,rationale,risk
0,Junk rows above header,rows before first row containing 'customer_id',1,dropped,"title text, not data",none
1,Columns outside data dictionary,column name not in data dictionary,2,"dropped ['aging', '<empty>']",'Aging' duplicates tenure_years (rebuilt from ...,none - tenure band recomputed
2,Non-customer rows (pasted summary/pivot tables),customer_id not matching ^CUST[0-9]+$,18,dropped,"they are report totals, not customers; would c...",none
3,Numbers stored as text with thousand separators,"value contains ','",9895,"separators stripped, cast to number","Indian-style '1,18,776' and '4,485' are numbers",none
4,Exact duplicate rows,all columns identical,0,none found,same customer counted twice,none
5,Same customer_id with different values,duplicated customer_id,0,none found,one row per customer is required for ranking,may keep the older record
6,Unrecognised/missing 'gender',"value not in ['Male', 'Female']",0,none found,"explicit category, not a guess",none
7,Unrecognised/missing 'region',"value not in ['North', 'South', 'East', 'West'...",0,none found,"explicit category, not a guess",none
8,Unrecognised/missing 'customer_segment',"value not in ['Mass Market', 'Mass Affluent', ...",0,none found,"explicit category, not a guess",none
9,Missing values (all numeric fields),isna() count,0,none found - logic kept for hidden test file,NaN,NaN


In [4]:
pd.read_csv('../outputs/data_quality_scorecard.csv')

,dimension,before,after
0,Rows that are real customers,10000/10020 lines,10000
1,"Completeness (non-null cells, data dictionary ...",1.0000,1.0000
2,Numeric columns stored as text,"yes (balance, e.g. '4,485')",no (all float/int)
3,Validity (rows within all domain ranges),1.0000,1.0000
4,Uniqueness (unique customer_id / rows),1.0000,1.0000
5,Consistency (rows with no logic flag),not measured,0.9764
6,Redundant/undocumented columns,Aging + empty column,removed; tenure_band rebuilt


In [5]:
pd.read_csv('../outputs/data_profile.csv', index_col=0).round(2)

,count,mean,std,min,25%,50%,75%,max,n_unique
age,10000.0,41.75,12.63,18.0,33.00,42.0,50.00,85.0,68
tenure_years,10000.0,5.46,5.42,0.1,1.60,3.8,7.70,32.0,290
monthly_income,10000.0,6766.33,6187.35,1500.0,3347.75,4593.5,7680.75,50048.0,6170
avg_monthly_balance,10000.0,34245.38,82101.65,430.0,3058.75,6924.0,26866.75,1228982.0,8051
num_products,10000.0,2.26,1.04,1.0,1.00,2.0,3.00,6.0,6
has_credit_card,10000.0,0.52,0.50,0.0,0.00,1.0,1.00,1.0,2
has_loan,10000.0,0.27,0.44,0.0,0.00,0.0,1.00,1.0,2
has_investment_product,10000.0,0.13,0.33,0.0,0.00,0.0,0.00,1.0,2
digital_engagement_score,10000.0,54.65,19.29,0.0,41.60,54.6,67.80,100.0,934
branch_visits_last_year,10000.0,3.70,2.17,0.0,2.00,3.0,5.00,14.0,15
